# 自由线程实践

学习目标：区分自由线程构建与实际 GIL 状态，在同版本解释器中检查线程安全、扩展导入和计算性能。

前置知识：对象引用、线程与锁、线程池、异常处理、上下文管理器、子进程、性能测量，以及 C 扩展的基本兼容条件。

运行环境：Notebook 使用 Python 3.12 内核。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

Notebook 启动临时 CPython 3.14.7 子进程完成实验；运行包适用于 Windows x64。

首次实验会从 NuGet 下载官方常规版和自由线程版运行包，共约 31.4 MB；需要网络与临时磁盘空间。包内容校验后才解压，实验结束即清理，不注册解释器或修改系统 PATH。重新执行下载单元会再次下载。

配套脚本：位于 [scripts/35-free-threading/](scripts/35-free-threading/)。

（1）[experiment.py](scripts/35-free-threading/experiment.py)：在独立进程中观察 GIL、扩展导入、共享更新与固定计算负载。

（2）[runtime_tools.py](scripts/35-free-threading/runtime_tools.py)：临时准备两个固定版本运行包，启动并等待实验进程，退出后清理。

（3）[tests/](scripts/35-free-threading/tests/)：检查计算边界、并发结果、测量轮次及解压边界。

## 1 从结果确定的小任务开始

本章用显式 Python 循环计算一段整数的平方和。bounds 是 (start, stop)，要求两个端点为普通整数且满足 0 ≤ start ≤ stop；start 包含在内，stop 不包含。

性能分析章节介绍测量方法，本章把解释器构建和 GIL 状态作为新的实验条件。先在当前内核核对算法，不把这一步当作自由线程实验。

In [1]:
import inspect
from pathlib import Path
import sys

lesson_dir = Path("scripts/35-free-threading").resolve()
original_path = sys.path.copy()
original_bytecode = sys.dont_write_bytecode
try:
    sys.dont_write_bytecode = True
    sys.path.insert(0, str(lesson_dir))
    import experiment
    import runtime_tools
finally:
    sys.path[:] = original_path
    sys.dont_write_bytecode = original_bytecode

print(inspect.getsource(experiment.square_total))  # 显示区间校验与逐项平方累加的源码，不执行该函数。
print(experiment.square_total((0, 4)))  # 14：0² + 1² + 2² + 3²。
print(experiment.square_total((3, 6)))  # 50：3² + 4² + 5²。

def square_total(bounds: tuple[int, int]) -> int:
    """累计左闭右开非负整数区间内的平方，使用显式 Python 循环。"""
    start, stop = bounds
    if type(start) is not int or type(stop) is not int:
        raise ValueError("区间端点必须是普通整数")
    if not 0 <= start <= stop:
        raise ValueError("区间必须满足 0 <= start <= stop")
    total = 0
    for number in range(start, stop):
        total += number * number
    return total

14
50


## 2 构建能力与运行状态分别判断

Python 3.13 开始提供自由线程构建，3.14 已正式支持，但常规构建仍带有 GIL。自由线程构建可以在启动时重新启用 GIL，因此不能只看版本或文件名。

| API／选项 | 中文名称／含义 |
| --- | --- |
| sysconfig.get_config_var("Py_GIL_DISABLED") | 构建能力；等于 1 表示支持自由线程 |
| sys._is_gil_enabled() | 当前进程是否启用了 GIL |
| -X gil=1 | 启动自由线程解释器时明确开启 GIL |
| -X gil=0 | 明确关闭 GIL；不能把不兼容扩展自动变成线程安全代码 |

本例使用同为 3.14.7 的常规构建、自由线程构建开启 GIL、自由线程构建默认关闭 GIL 三组条件。最后一组保留默认的扩展兼容处理，没有强制关闭保护机制。

配套实验一次收集状态、共享更新和计时结果，后面逐项阅读。计时输入固定为 8 段，每段 100000 个整数，每条执行路径测量 3 轮；下载和进程启动不计入算法耗时。

In [2]:
cases = [
    ("常规构建", "regular", False),
    ("自由线程构建，GIL 开启", "free_threaded", True),
    ("自由线程构建，GIL 关闭", "free_threaded", False),
]
comparisons = {}
# 下载并展开一次，三组子进程共用临时运行包；离开 with 后统一清理。
with runtime_tools.temporary_runtimes() as runtimes:
    runtime_paths = list(runtimes.values())
    # 每组都新启动解释器，把构建类型与 GIL 开关作为两个独立变量。
    for label, runtime_name, enable_gil in cases:
        comparisons[label] = runtime_tools.run_experiment(
            runtimes[runtime_name],
            enable_gil=enable_gil,
            items_per_job=100000,
            repeats=3,
        )
        state = comparisons[label]["state"]
        print(label, state["version"])  # 按 cases 次序显示三组名称；本章固定运行包版本为 3.14.7。
        print("构建支持自由线程：", state["free_threaded_build"])
        print("导入 CSV 前 GIL 开启：", state["gil_before_csv"])

# 三组预期分别为 False/True、True/True、True/False。
# 这些布尔值来自各实验进程，不能用当前 3.12 内核的状态代替。
assert comparisons["自由线程构建，GIL 关闭"]["state"]["gil_before_csv"] is False
assert all(not path.exists() for path in runtime_paths)

常规构建 3.14.7
构建支持自由线程： False
导入 CSV 前 GIL 开启： True


自由线程构建，GIL 开启 3.14.7
构建支持自由线程： True
导入 CSV 前 GIL 开启： True


自由线程构建，GIL 关闭 3.14.7
构建支持自由线程： True
导入 CSV 前 GIL 开启： False


## 3 扩展导入后再次检查 GIL

未声明自由线程支持的 C 扩展可能在导入时发出警告并重新开启 GIL。安装成功、导入成功、运行时没有 GIL、业务线程安全是不同条件。

本例观察标准库 csv 使用的 _csv 扩展。CPython 3.14.7 的 _csv 源码声明 Py_mod_gil 为 Py_MOD_GIL_NOT_USED；这里还实际读取两行记录，并比较导入前后状态。这个结果只支持本例扩展，不能推广到所有第三方包。

自由线程的扩展需要对应构建的二进制产物，3.14 的自由线程构建不支持 Limited C API／Stable ABI。不能把常规解释器的扩展文件直接复制过来作为兼容方案。

In [3]:
for label, report in comparisons.items():
    state = report["state"]
    print(
        label,
        "导入前已加载：", state["csv_was_loaded"],
        "GIL：", state["gil_before_csv"], "->", state["gil_after_csv"],
    )
    assert state["gil_before_csv"] == state["gil_after_csv"]
    assert state["csv_rows"] == [["name", "minutes"], ["python", "30"]]
print(comparisons["自由线程构建，GIL 关闭"]["state"]["csv_rows"])  # [['name', 'minutes'], ['python', '30']]；CSV 字段仍是字符串。
# 本例 _csv 没有重新开启 GIL；解析结果仍然正确。
# run_experiment 原样显示兼容警告；实际非零退出才由子进程异常传播。

常规构建 导入前已加载： False GIL： True -> True
自由线程构建，GIL 开启 导入前已加载： False GIL： True -> True
自由线程构建，GIL 关闭 导入前已加载： False GIL： False -> False
[['name', 'minutes'], ['python', '30']]


## 4 没有 GIL 仍然需要业务锁

dict、list 等内置容器在自由线程构建中使用内部锁保护部分操作，但这不把多个操作自动合成一个业务事务。线程章节中的“读取、加一、写回”仍需要同一把锁覆盖完整过程。

配套 compare_counters 先让两项工作读到同一个旧值，再由 Barrier 放行，稳定展示一次丢失更新；随后把完整更新放在锁内。屏障位于锁外，等待有超时，工作异常通过 Future.result 传播。

In [4]:
print(inspect.getsource(experiment.compare_counters))  # 显示屏障固定竞态、锁覆盖完整更新的两段对照源码。
for label, report in comparisons.items():
    print(label, report["counter"])
    assert report["counter"] == {"unlocked": 1, "locked": 2}
# 三组都应观察到 1 与 2：GIL 状态改变没有消除业务竞态。
# 两种路径退出线程池后才读取结果，不依赖随机休眠触发错误。

def compare_counters() -> dict[str, int]:
    """固定一次丢失更新，再比较同一业务操作使用锁时的结果。"""
    # 1. 两个任务都先读到零，再由屏障同时放行写回。
    counter = {"value": 0}
    barrier = threading.Barrier(2, timeout=5)

    def increment_unlocked() -> None:
        previous = counter["value"]
        barrier.wait()
        counter["value"] = previous + 1

    with ThreadPoolExecutor(max_workers=2) as executor:
        tasks = [executor.submit(increment_unlocked) for _ in range(2)]
        for task in tasks:
            task.result(timeout=10)
    unlocked = counter["value"]

    # 2. 会合仍在锁外，锁覆盖完整的读取、计算、写回。
    counter = {"value": 0}
    barrier = threading.Barrier(2, timeout=5)
    lock = threading.Lock()

    def increment_locked() -> None:
        barrier.wait()
        with lock:
            previous = counter["value"]
            counter["value"] = previous + 1

    with ThreadPoolExecutor(max_workers=2) as executor:
        tasks = [executor.submit(increment_locked) for _ in range(2)]
        for task in tasks:
    

## 5 让每个任务产生独立结果

另一种减少共享状态的方法是拆分输入，让每个工作函数使用自己的局部变量，最后由协调方收集结果。本章平方和的每一段只返回一个整数，不修改共享合计。

run_batch 的 workers=0 是本例约定，表示顺序调用；正数表示线程池上限。Executor.map 按输入顺序提供结果；with 等待工作结束并关闭线程池。这里输入只有八段，不涉及无限任务提交。

In [5]:
print(inspect.getsource(experiment.run_batch))  # 显示 workers=0 的顺序分支与保持输入次序的线程池分支。
small_bounds = [(3, 6), (0, 0), (0, 4), (3, 6)]
print(experiment.run_batch(small_bounds, workers=2))  # [50, 0, 14, 50]
assert experiment.run_batch(small_bounds, workers=0) == [50, 0, 14, 50]
# 此单元检查任务拆分的语义；三种 3.14 条件已经运行了相同的函数。

def run_batch(bounds: list[tuple[int, int]], workers: int) -> list[int]:
    """按输入顺序返回每段平方和；workers=0 表示顺序调用。"""
    if type(workers) is not int or workers < 0:
        raise ValueError("workers 必须是非负普通整数")
    if workers == 0:
        return [square_total(item) for item in bounds]
    with ThreadPoolExecutor(max_workers=workers) as executor:
        return list(executor.map(square_total, bounds, timeout=30))

[50, 0, 14, 50]


### 5.1 任务异常必须回到调用方

结果收集不能只检查线程是否结束。map 对应工作函数抛出异常时，在取出该项结果处重新抛出；本例保留这个错误，退出线程池时完成清理。

下面第二段的起点大于终点，违反本章区间约定，不应被悄悄当作零。

In [6]:
try:
    experiment.run_batch([(0, 4), (6, 3)], workers=2)
except ValueError as error:
    print(type(error).__name__, str(error))
    # ValueError 区间必须满足 0 <= start <= stop；没有成功结果列表。
else:
    raise AssertionError("无效工作区间未被拒绝")

ValueError 区间必须满足 0 <= start <= stop


## 6 明确计时范围与工作量

每组先预热顺序路径和 1、2、4 线程路径，再按这四条路径循环测量三轮。每轮都处理相同的八段输入，不随线程数增加工作量。

perf_counter 测量经过时间。计时包含 Python 计算、结果收集，以及线程池创建和关闭；不包含输入准备、正确性断言、打印、下载或解释器启动。本例没有关闭自动垃圾回收。

每段结果用独立的平方和公式核对，再检查完整合计。设 n 为整数个数，0 到 n−1 的平方和为 n(n−1)(2n−1)/6；实际被测函数仍逐项循环，公式只用于验证。

In [7]:
for label, report in comparisons.items():
    measured = report["benchmark"]
    print(label)
    # 每组均显示：任务数 8、每任务整数数 100000、轮次 3。
    print(
        "任务数：", measured["jobs"],
        "每任务整数数：", measured["items_per_job"],
        "轮次：", measured["repeats"],
    )
    assert measured["expected"] == 170666346666800000
    for row in measured["rows"]:
        milliseconds = [value * 1000 for value in row["seconds"]]
        print(row["workers"], [round(value, 3) for value in milliseconds])
# 每行第一项为 workers，0 表示顺序版；后面是三轮毫秒值。
# 全部保存实际测量值，不设置“线程版必须更快”的通过条件。

常规构建
任务数： 8 每任务整数数： 100000 轮次： 3
0 [218.944, 238.651, 217.566]
1 [236.385, 232.108, 197.995]
2 [235.302, 280.657, 189.848]
4 [282.875, 249.17, 232.079]
自由线程构建，GIL 开启
任务数： 8 每任务整数数： 100000 轮次： 3
0 [156.77, 163.99, 162.482]
1 [183.314, 129.051, 162.361]
2 [143.877, 128.367, 154.294]
4 [168.651, 150.623, 179.908]
自由线程构建，GIL 关闭
任务数： 8 每任务整数数： 100000 轮次： 3
0 [232.525, 262.243, 267.216]
1 [264.533, 255.045, 239.587]
2 [115.495, 149.15, 134.431]
4 [85.204, 103.896, 113.746]


### 6.1 同一组内比较线程扩展收益

用“同组顺序版最小耗时 ÷ 同组线程版最小耗时”计算比值，大于 1 表示这些试次中线程版的最小耗时更短。最小值帮助观察较少干扰的试次，不等于用户平均等待时间。

线程启动、调度、任务大小与机器负载都会影响结果；四个线程也不保证恰好占满四个核心。这里保留完整轮次，再读取比值，不能只选择一次最有利的测量。

In [8]:
for label, report in comparisons.items():
    rows = report["benchmark"]["rows"]
    serial_seconds = min(rows[0]["seconds"])
    print(label)
    for row in rows[1:]:
        ratio = serial_seconds / min(row["seconds"])
        print(f'{row["workers"]} 线程：顺序最小值 / 线程最小值 = {ratio:.2f}')
# 每个比值的分子和分母来自同一组构建与 GIL 状态。
# 本次比值仅适用于这些输入和当前机器，不能作为通用提速承诺。

常规构建
1 线程：顺序最小值 / 线程最小值 = 1.10
2 线程：顺序最小值 / 线程最小值 = 1.15
4 线程：顺序最小值 / 线程最小值 = 0.94
自由线程构建，GIL 开启
1 线程：顺序最小值 / 线程最小值 = 1.21
2 线程：顺序最小值 / 线程最小值 = 1.22
4 线程：顺序最小值 / 线程最小值 = 1.04
自由线程构建，GIL 关闭
1 线程：顺序最小值 / 线程最小值 = 0.97
2 线程：顺序最小值 / 线程最小值 = 2.01
4 线程：顺序最小值 / 线程最小值 = 2.73


### 6.2 跨构建比较还包含实现差异

自由线程构建为并发增加了不同的对象管理与同步成本，单线程也可能有额外开销。即使开启 GIL，自由线程构建也不会因此变成与常规构建完全相同的实现。

下面都取顺序路径，比较两种自由线程状态相对常规构建的耗时。若要研究其他输入规模、I/O 或原生库，应另设工作负载，不能沿用本例整数循环的结论。

In [9]:
regular_time = min(
    comparisons["常规构建"]["benchmark"]["rows"][0]["seconds"]
)
for label in ["自由线程构建，GIL 开启", "自由线程构建，GIL 关闭"]:
    serial_time = min(comparisons[label]["benchmark"]["rows"][0]["seconds"])
    print(f"{label}：顺序耗时 / 常规构建顺序耗时 = {serial_time / regular_time:.2f}")
# 比值大于 1 表示本次顺序路径耗时更长；这里没有混入 1 线程池的启动成本。

自由线程构建，GIL 开启：顺序耗时 / 常规构建顺序耗时 = 0.72
自由线程构建，GIL 关闭：顺序耗时 / 常规构建顺序耗时 = 1.07


## 7 共享对象与资源边界仍要明确

不要让多个线程并发推进同一个迭代器；在自由线程构建中可能出现重复或遗漏。优先拆分输入并为每项工作创建自己的迭代状态。

读取另一个正在执行的线程的 frame.f_locals 也不安全，不应用来制作“观察线程内部”的实验。对象回收时机不能代替文件或连接的显式关闭。

本章只保留 JSON 结果。每次实验进程退出后才删除临时解释器；线程池由脚本内的 with 清理，Notebook 内核没有加载临时构建的扩展文件。

In [10]:
print(all(not path.exists() for path in runtime_paths))  # True：运行目录已清理。
for label, report in comparisons.items():
    assert report["state"]["version"] == "3.14.7"
    assert report["counter"]["locked"] == 2
print("三组结果仍可读取")  # 删除运行文件不会删除内存中的结果字典。
# 不把 Python 3.14 的运行包长期放入 scripts，也不需要卸载共享环境的包。

True
三组结果仍可读取


## 8 用测试保住比较前提

配套测试检查空区间、端点错误、结果顺序与重复、工作异常、锁内更新，以及计时轮数。运行包解压还检查内容哈希和路径边界。

这些测试在共享环境中检查实验代码的行为；前面的三组子进程才是自由线程实测。两者通过分别说明不同问题，不把测试报告中的通过数当作性能收益。

In [11]:
import os
import subprocess
from tempfile import TemporaryDirectory

test_env = os.environ.copy()
test_env.update({
    "PYTHONPATH": str(lesson_dir),
    "PYTHONDONTWRITEBYTECODE": "1",
    "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1",
    "PYTEST_ADDOPTS": "",
})
# 测试只验证宿主辅助代码和实验函数，真实构建对照已在前面运行。
with TemporaryDirectory() as directory:
    checked = subprocess.run(
        [
            sys.executable, "-B", "-m", "pytest", str(lesson_dir / "tests"),
            "-q", "-p", "no:cacheprovider",
            "--basetemp", str(Path(directory) / "pytest"),
        ],
        env=test_env, capture_output=True, encoding="utf-8",
        check=True, timeout=30,
    )
print(checked.stdout, end="")  # 24 passed；耗时随实际运行变化。
assert "24 passed" in checked.stdout
# 应实际运行 24 项测试；测试临时目录已清理。

........................                                                 [100%]
24 passed in 0.74s


## 本章小结

（1）构建支持自由线程与进程当前关闭 GIL 是两个判断；扩展导入后还要确认实际状态。

（2）自由线程不消除业务竞态。用锁覆盖完整更新，或让任务拥有独立输入与结果。

（3）比较性能时保持版本、输入和计时边界一致，同时检查顺序、完整结果和异常传播。

（4）先保存各轮实际值，再解释同组线程收益和跨构建成本；不要依赖固定加速倍数或对象回收时机。

自查：如果扩展导入重新开启了 GIL，或者任务数随线程数变化，原有对照还在回答同一个问题吗？

## 练习

（1）先预测下面三组布尔值，再运行核对。分别解释“构建支持”与“当前开启”的含义；核对标准是预测与实测结果一致，且能说明为什么只看版本号不够。

In [12]:
for label, report in comparisons.items():
    state = report["state"]
    print(label, state["free_threaded_build"], state["gil_after_csv"])
# 先写预测，再核对两个状态字段；不要从名称里的 t 直接推断运行状态。

常规构建 False True
自由线程构建，GIL 开启 True True
自由线程构建，GIL 关闭 True False


（2）在新的 temporary_runtimes 管理范围内，仅用自由线程构建执行 benchmark 场景，把每段整数数改为 3、轮次改为 2。检查共有 8 个任务、合计为 4324、四条路径各有两个非负耗时。

确认 GIL 仍关闭，退出 with 后解释器路径不存在。这个小输入用于检查实验结构，不要求线程版更快；重新准备运行包会再次下载。

In [13]:
exercise_items_per_job = 3
exercise_repeats = 2
# 用 run_experiment 的 scenario="benchmark" 指定观察范围。
# 保存实际报告，核对完整条件后退出 with；不要断言固定耗时。

（3）为 run_batch 编写两个临时测试：输入 [(2, 5), (2, 5), (0, 0)] 时，顺序与两线程均应返回 [29, 29, 0]；其中一段改为 (5, 2) 时，两线程调用应抛出 ValueError。

用共享环境实际运行 pytest，检查两项测试通过，临时文件已删除。另说明若改为让所有任务直接写同一个合计变量，为什么现有结果正确性仍不足以代替同步设计。

In [14]:
exercise_bounds = [(2, 5), (2, 5), (0, 0)]
# 在 TemporaryDirectory 中创建测试文件，使用独立写出的期望值。
# 异常测试保留原业务函数，不能通过替换 square_total 伪造失败。

## 参考与引用来源

| 网站 | 资料与知识点定位 |
| --- | --- |
| Python 官方文档 | 3.14：[自由线程安装与构建](https://docs.python.org/3.14/howto/free-threading-python.html#installation)、[构建能力与运行状态](https://docs.python.org/3.14/howto/free-threading-python.html#identifying-free-threaded-python)、[运行时 GIL 与扩展导入](https://docs.python.org/3.14/howto/free-threading-python.html#the-global-interpreter-lock-in-free-threaded-python)、[线程安全与内部锁的边界](https://docs.python.org/3.14/howto/free-threading-python.html#thread-safety)、[迭代器、frame 与单线程成本](https://docs.python.org/3.14/howto/free-threading-python.html#known-limitations)、[3.14 正式支持自由线程](https://docs.python.org/3.14/whatsnew/3.14.html#free-threaded-python-is-officially-supported)；[Windows 官方 NuGet 包](https://docs.python.org/3.14/using/windows.html#the-nuget-org-packages)、[自由线程 NuGet 包](https://docs.python.org/3.14/using/windows.html#free-threaded-packages)、[-X 选项](https://docs.python.org/3.14/using/cmdline.html#cmdoption-X)、[-I](https://docs.python.org/3.14/using/cmdline.html#cmdoption-I)、[-S](https://docs.python.org/3.14/using/cmdline.html#cmdoption-S)；[扩展的支持声明](https://docs.python.org/3.14/howto/free-threading-extensions.html#module-initialization)、[专用构建与二进制产物](https://docs.python.org/3.14/howto/free-threading-extensions.html#building-extensions-for-the-free-threaded-build)、[Limited C API 与 Stable ABI 限制](https://docs.python.org/3.14/howto/free-threading-extensions.html#limited-c-api-and-stable-abi)、[csv.reader](https://docs.python.org/3.14/library/csv.html#csv.reader)；[Barrier](https://docs.python.org/3.14/library/threading.html#threading.Barrier)、[锁](https://docs.python.org/3.14/library/threading.html#lock-objects)、[Executor.map](https://docs.python.org/3.14/library/concurrent.futures.html#concurrent.futures.Executor.map)、[线程池关闭](https://docs.python.org/3.14/library/concurrent.futures.html#concurrent.futures.Executor.shutdown)、[任务异常传播](https://docs.python.org/3.14/library/concurrent.futures.html#concurrent.futures.Future.result)、[perf_counter](https://docs.python.org/3.14/library/time.html#time.perf_counter)。宿主辅助代码使用 3.12：[临时目录](https://docs.python.org/3.12/library/tempfile.html#tempfile.TemporaryDirectory)、[子进程](https://docs.python.org/3.12/library/subprocess.html#subprocess.run)、[下载响应](https://docs.python.org/3.12/library/urllib.request.html#urllib.request.urlopen)、[SHA-256](https://docs.python.org/3.12/library/hashlib.html#hashlib.sha256)、[ZIP 提取边界](https://docs.python.org/3.12/library/zipfile.html#zipfile.ZipFile.extractall)。 |
| GitHub 上的 CPython 源码 | v3.14.7：[csv 使用 _csv 接口](https://github.com/python/cpython/blob/v3.14.7/Lib/csv.py#L68-L72)、[_csv 声明无需 GIL](https://github.com/python/cpython/blob/v3.14.7/Modules/_csv.c#L1844-L1846)。 |
| NuGet 官方包页面 | [python 3.14.7](https://www.nuget.org/packages/python/3.14.7)、[python-freethreaded 3.14.7](https://www.nuget.org/packages/python-freethreaded/3.14.7)。配套脚本固定下载版本及本次取得内容的 SHA-256，不把哈希值称为独立的发布者签名。 |
| Microsoft Learn | [NuGet V3 包内容 API](https://learn.microsoft.com/en-us/nuget/api/package-base-address-resource#download-package-content-nupkg)，用于取得固定版本的运行包。 |
| pytest 文档 | pytest 9.1.1：[断言与预期异常](https://docs.pytest.org/en/stable/how-to/assert.html#assertions-about-expected-exceptions)、[参数化](https://docs.pytest.org/en/stable/how-to/parametrize.html)、[临时目录](https://docs.pytest.org/en/stable/how-to/tmp_path.html)。 |